# Taxonomic Profiling of Shotgun Metagenomes

**Tier 3 — Applied Bioinformatics | Module 26 · Notebook 1**

*Prerequisites: Module 01 (NGS Fundamentals), Module 04 (Microbial Diversity)*

---

**By the end of this notebook you will be able to:**
1. Explain the difference between 16S amplicon and shotgun metagenomics
2. Decontaminate metagenomic reads by removing host sequences
3. Classify reads taxonomically with Kraken2
4. Re-estimate species abundances with Bracken
5. Visualize community composition with Krona and bar charts



**Key resources:**
- [QIIME2 tutorials](https://docs.qiime2.org/2024.10/tutorials/)
- [Galaxy Training — Metagenomics](https://training.galaxyproject.org/training-material/topics/metagenomics/)
- [Kraken2 documentation](https://github.com/DerrickWood/kraken2)

---

[< Previous: Isoform Analysis with Long Reads](../25_Long_Read_Sequencing/03_isoform_analysis.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Functional Annotation of Metagenomes >](02_functional_annotation.ipynb)

---

## 1. Shotgun vs 16S Metagenomics

Two major strategies exist for characterizing microbial communities:

| Feature | 16S rRNA amplicon | Shotgun metagenomics |
|---|---|---|
| Target | 16S rRNA gene (V3-V4 region) | All DNA in the sample |
| Taxonomic resolution | Genus level typical; species sometimes | Species and strain level |
| Functional information | No | Yes (gene/pathway content) |
| Host contamination | N/A (bacteria-specific amplification) | Major issue (requires decontamination) |
| Cost per sample | Low (~$30–80) | Higher (~$200–500) |
| Bioinformatics complexity | Moderate (QIIME2, DADA2) | High (multi-step pipeline) |
| Database bias | 16S database coverage | Requires organisms in reference DB |

**When to use shotgun:** Research questions requiring species/strain resolution, functional gene content, AMR gene detection, or viral metagenomics.

### Shotgun Metagenomics Pipeline Overview

```
Raw FASTQ → QC (fastp) → Host decontamination (Bowtie2/hg38)
  → Taxonomic profiling (Kraken2 + Bracken)
  → Functional profiling (HUMAnN3)
  → Assembly (MEGAHIT)
  → Binning (MetaBAT2)
  → MAG QC (CheckM)
```

## 2. Host Read Removal

Shotgun metagenomic samples from human gut, skin, or blood contain significant host (human) DNA. If not removed, host reads inflate computational cost and cause false taxonomic assignments (reads misassigned to bacteria with similar sequence content).

**Tool:** Bowtie2 with `--un-conc-gz` to output read pairs that did NOT map to the host.

```bash
# Build or download hg38 Bowtie2 index (one-time setup)
# bowtie2-build hg38.fa hg38_index/hg38

# Decontaminate: align to human genome, keep unmapped reads
bowtie2 -x hg38_index/hg38 \
    -1 sample_R1.fastq.gz \
    -2 sample_R2.fastq.gz \
    -p 8 \
    --very-sensitive \
    --un-conc-gz decontam_%.fastq.gz \  # unmapped pairs → decontam_1.fastq.gz, decontam_2.fastq.gz
    > /dev/null  # discard human-mapped SAM output

# Report host content
echo "Host contamination: $(cat contamination_rate.txt)%"
```

**Typical host contamination rates:**
- Gut metagenome: 1–10% human reads
- Bronchoalveolar lavage: 40–80% human reads
- Skin: 10–30% human reads
- PBMC or blood: 60–95% human reads

**Alternative for known pathogens:** Use the **Decontam** R package to remove contaminants based on inverse correlation with DNA concentration across negative controls.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# Simulate host contamination profiles across sample types
sample_types = {
    'Gut (stool)':     rng.beta(2, 20, size=20),        # low contamination
    'Skin swab':       rng.beta(3, 10, size=20),        # moderate
    'Oral rinse':      rng.beta(3, 15, size=20),        # moderate-low
    'BAL (lung)':      rng.beta(6, 4,  size=20),        # high
    'PBMC (blood)':    rng.beta(10, 2, size=20),        # very high
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Box plot
data = list(sample_types.values())
labels = list(sample_types.keys())
bp = axes[0].boxplot(data, labels=labels, patch_artist=True, notch=False)
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[0].set_ylabel('Host read fraction')
axes[0].set_title('Host contamination by sample type')
axes[0].tick_params(axis='x', rotation=25)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

# Before/after decontamination: simulated total reads
n_samples = 10
total_before = rng.integers(15_000_000, 40_000_000, size=n_samples)
host_fracs   = rng.beta(2, 15, size=n_samples)
total_after  = (total_before * (1 - host_fracs)).astype(int)

x = np.arange(n_samples)
axes[1].bar(x - 0.2, total_before / 1e6, width=0.4, label='Before', color='#C44E52', alpha=0.8)
axes[1].bar(x + 0.2, total_after  / 1e6, width=0.4, label='After',  color='#4C72B0', alpha=0.8)
axes[1].set_xlabel('Sample')
axes[1].set_ylabel('Read pairs (millions)')
axes[1].set_title('Read count before/after host removal')
axes[1].legend()
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'S{i+1}' for i in range(n_samples)])

plt.tight_layout()
plt.show()

print(f'Mean host fraction: {host_fracs.mean():.1%}')
print(f'Mean reads retained: {(1 - host_fracs).mean():.1%}')


## 3. Taxonomic Classification with Kraken2

**Kraken2** classifies metagenomic reads by assigning each read to the lowest common ancestor (LCA) of all genomes in its reference database that share k-mer sequences with the read. Key parameters:

- **Database**: standard (bacteria + archaea + viruses + human), PlusPF (adds protozoa + fungi), or custom. Larger databases improve sensitivity but require more RAM (standard: ~55 GB; PlusPF: ~100 GB).
- `--confidence 0.1`: minimum fraction of k-mers classified to a node; helps reduce false assignments (default 0 = report everything)
- `--minimum-hit-groups 3`: minimum number of minimizer groups in the database before a read is assigned (reduces spurious assignments)

```bash
# Classify decontaminated reads
kraken2 \
    --db kraken2_db/ \
    --paired \
    --gzip-compressed \
    --threads 16 \
    --confidence 0.1 \
    --minimum-hit-groups 3 \
    --report kraken2_report.txt \
    --output kraken2_output.txt \
    decontam_1.fastq.gz decontam_2.fastq.gz
```

**Kraken2 report columns** (`kraken2_report.txt`):
1. Percentage of reads classified to this clade
2. Reads classified at this exact taxon
3. Reads classified within this clade (includes sub-taxa)
4. Rank code (S=species, G=genus, F=family, O=order, C=class, P=phylum, K=kingdom, R=root)
5. NCBI taxonomy ID
6. Indented scientific name

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)

# Simulate a Kraken2 report for a gut metagenome sample
species = [
    'Bacteroides vulgatus',      'Faecalibacterium prausnitzii',
    'Prevotella copri',          'Bifidobacterium longum',
    'Ruminococcus gnavus',       'Akkermansia muciniphila',
    'Bacteroides uniformis',     'Blautia obeum',
    'Lachnospiraceae bacterium', 'Eubacterium rectale',
    'Clostridium bolteae',       'Streptococcus salivarius',
    'Collinsella aerofaciens',   'Dorea longicatena',
    'Roseburia intestinalis',
]
n_species = len(species)

# Simulate classification counts (log-normal distributed)
read_counts = rng.lognormal(mean=10.5, sigma=1.8, size=n_species).astype(int)
read_counts = np.sort(read_counts)[::-1]
total_classified = read_counts.sum()
total_reads = int(total_classified / 0.75)  # ~75% classification rate

fractions = read_counts / total_reads * 100

kraken_df = pd.DataFrame({
    'Species':   species,
    'Reads':     read_counts,
    'Percentage': fractions,
}).sort_values('Reads', ascending=False)

print('=== Kraken2 Classification Summary ===')
print(f'Total reads processed:  {total_reads:>12,}')
print(f'Total classified:       {total_classified:>12,}  ({total_classified/total_reads:.1%})')
print(f'\nTop 10 species:')
print(kraken_df[['Species', 'Reads', 'Percentage']].head(10).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 6))
top10 = kraken_df.head(10)
colors = plt.cm.tab10(np.linspace(0, 1, 10))
bars = ax.barh(range(len(top10)), top10['Percentage'][::-1], color=colors)
ax.set_yticks(range(len(top10)))
ax.set_yticklabels(top10['Species'][::-1], fontsize=9)
ax.set_xlabel('Percentage of total reads (%)')
ax.set_title('Kraken2 top species — gut metagenome')
ax.axvline(1.0, color='red', linestyle=':', lw=1, label='1% threshold')
ax.legend()
plt.tight_layout()
plt.show()


## 4. Abundance Re-estimation with Bracken

**Bracken** (Bayesian Re-estimation of Abundance with KraKEN) addresses a fundamental limitation of Kraken2: reads that share k-mers with multiple species are assigned to their lowest common ancestor (LCA), inflating higher-taxon abundances. Bracken uses the k-mer distribution across all genomes in the database to **redistribute** these reads back to the most likely species level.

**Key parameters:**
- `-r 150`: read length (should match your actual sequencing read length)
- `-l S`: taxonomic level to estimate (`S`=Species, `G`=Genus, `F`=Family)
- `-t 10`: minimum reads threshold at the species level; species with fewer reads are excluded

```bash
# Species-level abundance re-estimation
bracken \
    -d kraken2_db/ \
    -i kraken2_report.txt \
    -o bracken_species.txt \
    -w bracken_species_report.txt \
    -r 150 \
    -l S \
    -t 10
```

**Bracken output columns** (`bracken_species.txt`):
- `name`: species name
- `taxonomy_id`: NCBI taxonomy ID
- `taxonomy_lvl`: level (S)
- `kraken_assigned_reads`: reads Kraken2 assigned directly to this species
- `added_reads`: reads redistributed down from higher-level LCA nodes
- `new_est_reads`: total = kraken_assigned_reads + added_reads
- `fraction_total_reads`: new_est_reads / total_reads (the abundance estimate)

The fraction_total_reads column is what you use for downstream diversity analysis and differential abundance testing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(2)

# Simulate Bracken output showing redistribution from LCA nodes
species = [
    'Bacteroides vulgatus',      'Faecalibacterium prausnitzii',
    'Prevotella copri',          'Bifidobacterium longum',
    'Ruminococcus gnavus',       'Akkermansia muciniphila',
    'Bacteroides uniformis',     'Blautia obeum',
    'Lachnospiraceae bacterium', 'Eubacterium rectale',
    'Clostridium bolteae',       'Streptococcus salivarius',
    'Collinsella aerofaciens',   'Dorea longicatena',
    'Roseburia intestinalis',
]

kraken_direct  = rng.lognormal(10.5, 1.8, size=len(species)).astype(int)
# Bracken adds redistributed reads (from LCA nodes)
added_reads    = (kraken_direct * rng.lognormal(-0.3, 0.8, len(species))).astype(int)
added_reads    = np.abs(added_reads)
total_est      = kraken_direct + added_reads
total_all_reads = total_est.sum() * 4  # approximate total reads

bracken_df = pd.DataFrame({
    'Species':               species,
    'Kraken2_direct':        kraken_direct,
    'Bracken_added':         added_reads,
    'Total_est_reads':       total_est,
    'Fraction_total':        total_est / total_all_reads,
}).sort_values('Total_est_reads', ascending=False)

print('=== Bracken species-level abundance ===')
print(bracken_df[['Species', 'Kraken2_direct', 'Bracken_added', 'Fraction_total']].head(10).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Stacked bar: Kraken2 direct vs Bracken added
top10 = bracken_df.head(10).reset_index(drop=True)
x = np.arange(len(top10))
axes[0].barh(x, top10['Kraken2_direct'],        label='Kraken2 direct', color='#4C72B0', alpha=0.85)
axes[0].barh(x, top10['Bracken_added'],
             left=top10['Kraken2_direct'],       label='Bracken added',  color='#DD8452', alpha=0.85)
axes[0].set_yticks(x)
axes[0].set_yticklabels(top10['Species'], fontsize=8)
axes[0].set_xlabel('Read count')
axes[0].set_title('Kraken2 vs Bracken redistribution')
axes[0].legend()

# Relative abundance pie of top 7 + other
top7 = bracken_df.head(7)
other_frac = bracken_df.iloc[7:]['Fraction_total'].sum()
pie_labels = list(top7['Species']) + ['Other']
pie_vals   = list(top7['Fraction_total']) + [other_frac]
colors_pie = plt.cm.tab10(np.linspace(0, 1, 8))
axes[1].pie(pie_vals, labels=[s.split()[-1] for s in pie_labels],
            colors=colors_pie, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 8})
axes[1].set_title('Species-level relative abundance\n(Bracken estimates)')

plt.tight_layout()
plt.show()


## 4b. Alternative Approach: MetaPhlAn Marker-Gene Profiling

**MetaPhlAn** (Metagenomic Phylogenetic Analysis) uses a fundamentally different approach from Kraken2: instead of classifying every read by k-mer LCA, it aligns reads to a curated database of **clade-specific marker genes** — unique phylogenetic markers present in one species/clade but absent from all others.

**Comparison of approaches:**

| Feature | Kraken2 + Bracken | MetaPhlAn4 |
|---|---|---|
| Method | k-mer LCA classification | Marker gene alignment (Bowtie2) |
| Speed | Very fast (10+ Gb/min) | Moderate (~1 Gb/min) |
| RAM required | 55–100 GB (database) | ~3 GB (marker DB) |
| Sensitivity | High for taxa in DB | Lower for novel organisms |
| False positive rate | Higher (k-mer sharing) | Lower (marker-specific) |
| Relative abundance | Bracken re-estimation needed | Direct from marker coverage |
| Best for | Clinical/pathogen detection, broad survey | Human microbiome studies, HMP-compatible output |
| Output | Report + classification file | Merged abundance table (TSV) |
| Database | Comprehensive genome DB | ~5.1 M marker genes (SGB-based) |

**When to use which:**
- Use **Kraken2 + Bracken** when you need maximum sensitivity (detecting pathogens, viruses) or fast turnaround
- Use **MetaPhlAn4** for human microbiome studies where consistency with published cohorts (HMP, curatedMetagenomicData) matters, or when RAM is limited

```bash
# Install MetaPhlAn4 and download the marker database (~3 GB)
# conda install -c bioconda metaphlan

# Download marker database (one time)
metaphlan --install --bowtie2db metaphlan_db/

# Run taxonomic profiling on a sample
metaphlan decontam_1.fastq.gz,decontam_2.fastq.gz \
    --input_type fastq \
    --bowtie2db metaphlan_db/ \
    --nproc 8 \
    -o metaphlan_profile.txt \
    --bowtie2out metaphlan_mapped.bam

# Merge multiple samples into one abundance table (for cohort analysis)
merge_metaphlan_tables.py metaphlan_*.txt > merged_abundance_table.txt
```

**MetaPhlAn output format** (`metaphlan_profile.txt`):
```
#clade_name    clade_taxid    relative_abundance    coverage    estimated_number_of_reads_from_the_clade
k__Bacteria|p__Firmicutes|c__Clostridia|...|s__Faecalibacterium_prausnitzii    ...    12.34    ...
```
The relative_abundance column sums to 100 across all species. Use `--tax_lev s` to extract species-level only.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(10)

# Simulate MetaPhlAn output for 5 samples (gut metagenome)
# MetaPhlAn outputs relative abundances that sum to 100 per sample
species = [
    'k__Bacteria|...|s__Bacteroides_vulgatus',
    'k__Bacteria|...|s__Faecalibacterium_prausnitzii',
    'k__Bacteria|...|s__Prevotella_copri',
    'k__Bacteria|...|s__Bifidobacterium_longum',
    'k__Bacteria|...|s__Ruminococcus_gnavus',
    'k__Bacteria|...|s__Akkermansia_muciniphila',
    'k__Bacteria|...|s__Blautia_obeum',
    'k__Bacteria|...|s__Eubacterium_rectale',
]
n_sp, n_samp = len(species), 5

# Dirichlet samples for each condition
alpha = np.array([5, 4, 3, 2.5, 2, 1.5, 1, 1])  # unequal dominance
raw = rng.dirichlet(alpha, size=n_samp) * 100  # convert to percent

metaphlan_df = pd.DataFrame(raw, columns=species)
metaphlan_df.index = [f'Sample_{i+1}' for i in range(n_samp)]

print('=== MetaPhlAn relative abundance table (%) ===')
# Shorten species names for display
short_names = [s.split('|s__')[-1].replace('_', ' ') for s in species]
display_df = metaphlan_df.copy()
display_df.columns = short_names
print(display_df.round(2).to_string())
print(f'\nRow sums (should be 100): {metaphlan_df.sum(axis=1).round(1).values}')

# Compare Kraken2 vs MetaPhlAn relative abundances on the same sample
# MetaPhlAn tends to be more conservative (fewer false positives)
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(n_sp)
w = 0.4
kraken_abund = rng.dirichlet(alpha * 1.1) * 100
metaphlan_abund = rng.dirichlet(alpha * 0.9) * 100
ax.bar(x - w/2, kraken_abund,   width=w, label='Kraken2+Bracken', color='#4C72B0', alpha=0.85)
ax.bar(x + w/2, metaphlan_abund, width=w, label='MetaPhlAn4',       color='#DD8452', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(short_names, rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Relative abundance (%)')
ax.set_title('Kraken2+Bracken vs MetaPhlAn4 — same sample (simulated)')
ax.legend()
plt.tight_layout()
plt.show()

print('\nKey differences (simulated):')
for sp, k, m in zip(short_names, kraken_abund, metaphlan_abund):
    diff = k - m
    print(f'  {sp:<35} Kraken={k:5.1f}%  MetaPhlAn={m:5.1f}%  diff={diff:+.1f}%')

## 5. Community Composition Visualization

After running Bracken, you typically have a per-sample abundance table (`fraction_total_reads` per species). Visualizations for microbiome community structure:

1. **Stacked bar chart**: relative abundance of top N species across all samples
2. **Krona chart**: hierarchical interactive HTML visualization (`ktImportTaxonomy` from KronaTools)
3. **Alpha diversity**: within-sample diversity (Shannon index, Simpson's diversity, observed richness)
4. **Beta diversity**: between-sample differences (Bray-Curtis dissimilarity, then PCoA/NMDS)

### Alpha Diversity Metrics

- **Shannon index** H = -Σ p_i × ln(p_i): measures both richness and evenness; ranges 0 (one species) to ln(S) (perfectly even)
- **Simpson's index** D = Σ p_i²: probability that two random reads are from the same species; reported as 1-D (diversity) or 1/D (inverse)
- **Observed richness**: raw number of species with at least one read (sensitive to sequencing depth)

All metrics are sensitive to sequencing depth — use **rarefaction** (random subsampling to equal depth) before comparison across samples with different library sizes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import entropy

rng = np.random.default_rng(5)

# Simulate multi-sample Bracken output: 3 conditions, 4 reps each
n_samples = 12
n_species = 30
condition = ['Healthy'] * 4 + ['IBD'] * 4 + ['CRC'] * 4

# Dirichlet samples for each condition (different alpha = different compositions)
alpha_healthy = np.ones(n_species) * 0.8 + rng.exponential(1, n_species)
alpha_ibd     = np.ones(n_species) * 0.3 + rng.exponential(0.5, n_species)
alpha_crc     = np.ones(n_species) * 0.4 + rng.exponential(0.7, n_species)

# Simulate reduced diversity in IBD/CRC
alpha_ibd[:3] *= 5   # few dominant species
alpha_crc[:2] *= 8

compositions = np.vstack([
    rng.dirichlet(alpha_healthy, size=4),
    rng.dirichlet(alpha_ibd,     size=4),
    rng.dirichlet(alpha_crc,     size=4),
])  # shape: (12, 30)

species_names = [f'Species_{i+1:02d}' for i in range(n_species)]
comp_df = pd.DataFrame(compositions, columns=species_names)
comp_df.insert(0, 'Sample', [f'{c}_rep{i%4+1}' for i, c in enumerate(condition)])
comp_df.insert(1, 'Condition', condition)

# Alpha diversity
shannon = comp_df[species_names].apply(
    lambda row: entropy(row[row > 0]), axis=1
)
comp_df['Shannon'] = shannon

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Stacked bar (top 10 species)
top10_sp = comp_df[species_names].mean().nlargest(10).index.tolist()
colors10 = plt.cm.tab10(np.linspace(0, 1, 10))
bottom = np.zeros(n_samples)
for sp, color in zip(top10_sp, colors10):
    axes[0].bar(range(n_samples), comp_df[sp], bottom=bottom, label=sp, color=color, width=0.8)
    bottom += comp_df[sp].values
axes[0].set_xticks(range(n_samples))
axes[0].set_xticklabels(comp_df['Sample'], rotation=45, ha='right', fontsize=7)
axes[0].set_ylabel('Relative abundance')
axes[0].set_title('Community composition (top 10 species)')
axes[0].legend(loc='upper right', fontsize=6, ncol=2)

# Alpha diversity by condition
cond_colors = {'Healthy': '#55A868', 'IBD': '#C44E52', 'CRC': '#DD8452'}
for cond, col in cond_colors.items():
    mask = comp_df['Condition'] == cond
    axes[1].scatter(np.where(mask)[0], comp_df.loc[mask, 'Shannon'],
                    color=col, s=60, label=cond, zorder=3)
    y = comp_df.loc[mask, 'Shannon']
    axes[1].hlines(y.mean(), np.where(mask)[0].min(), np.where(mask)[0].max(),
                   colors=col, linestyles='--', lw=1.5)
axes[1].set_ylabel('Shannon index')
axes[1].set_title('Alpha diversity by condition')
axes[1].set_xticks(range(n_samples))
axes[1].set_xticklabels(comp_df['Sample'], rotation=45, ha='right', fontsize=7)
axes[1].legend()

# Shannon box plot per condition
conds = ['Healthy', 'IBD', 'CRC']
box_data = [comp_df.loc[comp_df['Condition'] == c, 'Shannon'].values for c in conds]
bp = axes[2].boxplot(box_data, labels=conds, patch_artist=True, notch=True)
for patch, (cond, col) in zip(bp['boxes'], cond_colors.items()):
    patch.set_facecolor(col); patch.set_alpha(0.75)
axes[2].set_ylabel('Shannon index')
axes[2].set_title('Alpha diversity comparison')

plt.tight_layout()
plt.show()

print('Mean Shannon index by condition:')
for cond in conds:
    vals = comp_df.loc[comp_df['Condition'] == cond, 'Shannon']
    print(f'  {cond:10s}: {vals.mean():.3f} ± {vals.std():.3f}')


## Summary

Shotgun metagenomic taxonomic profiling pipeline:

1. **Host decontamination** (Bowtie2 + hg38): removes human reads before classification; required for human-associated samples to prevent false assignments
2. **Kraken2 classification**: k-mer LCA matching against a reference database; fast (~GB/min) but assigns reads to LCA when k-mers are shared across species
3. **Bracken re-estimation**: redistributes LCA-assigned reads back to the most probable species level using k-mer coverage models; produces accurate relative abundance estimates
4. **MetaPhlAn4 (alternative)**: marker-gene alignment approach; lower RAM (~3 GB vs 55+ GB for Kraken2), fewer false positives, directly compatible with human microbiome databases (HMP, curatedMetagenomicData)
5. **Visualization**: stacked bar (composition), alpha diversity (Shannon, Simpson), beta diversity (Bray-Curtis PCoA)

**Kraken2 vs MetaPhlAn4 — when to choose:**
- **Kraken2 + Bracken**: maximum sensitivity (pathogens, viruses), fast turnaround, broad taxon coverage
- **MetaPhlAn4**: human microbiome research, RAM-limited environments, reduced false positives

**Database choice matters:** The standard Kraken2 database (bacteria + archaea + virus + human) misses fungi and protozoa. Use PlusPF for broader coverage. Custom databases can be built from novel genomes with `kraken2-build`.

**Confidence threshold:** Kraken2 `--confidence 0.1` significantly reduces false species assignments, especially from highly repetitive regions. Default (0) maximizes sensitivity but increases false positives.

Next: functional annotation with HUMAnN3 → [Notebook 2: Functional Annotation](./02_functional_annotation.ipynb)

---

[< Previous: Isoform Analysis with Long Reads](../25_Long_Read_Sequencing/03_isoform_analysis.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Functional Annotation of Metagenomes >](02_functional_annotation.ipynb)

---